In [41]:
import os
import pickle
import pandas as pd
import torch
from tqdm import tqdm

In [42]:
ROOT_DIR = os.path.join('/home', 'rlawlsgurjh', 'work', 'MPOCSR')

In [43]:
# TASK_NAME = 'DECIMER'
TASK_NAME = 'Drug_like'

if TASK_NAME == 'Drug_like':
    DATA_DIR = os.path.join(ROOT_DIR, '..', 'MolScribe', 'data', 'DECIMER', 'DECIMER')
else:
    DATA_DIR = os.path.join(ROOT_DIR, '..', 'MolScribe', 'data', 'original')


In [44]:
#filename = 'filtered_DECIMER_test'
#filename = 'filtered_DECIMER_train_train'
#filename = 'filtered_DECIMER_train_val'

filename = 'transform_rdkit'
#filename = 'transform_randepict'
#filename = 'transform_repaint'

csv_path = os.path.join(DATA_DIR, f'{filename}.csv')
save_path = os.path.join(ROOT_DIR, 'Data', TASK_NAME, f'{filename}.pkl')
os.makedirs(os.path.dirname(save_path), exist_ok=True)

In [45]:
map_path = os.path.join(ROOT_DIR, 'Data', '200w_word_map.pth')

In [46]:
def create_label_map(smiles, word_map, max_length):
    """SMILES 문자열을 토큰화하고 인덱스로 매핑"""
    # 특수 토큰 추가
    tokens = ['<start>']
    
    # SMILES를 문자 단위로 토큰화
    tokens.extend(list(smiles))
    tokens.append('<end>')
    
    # 토큰을 인덱스로 매핑
    label_map = []
    for token in tokens:
        if token in word_map:
            label_map.append(word_map[token])
        else:
            label_map.append(word_map['<unk>'])
    
    # 패딩 추가
    label_length = len(label_map)
    if len(label_map) < max_length:
        label_map.extend([word_map['<pad>']] * (max_length - len(label_map)))
    else:
        label_map = label_map[:max_length]
        label_length = max_length
            
    return label_map, label_length

In [47]:
def convert_to_pickle(TASK_NAME,csv_path, map_path, save_path, max_length=100):   
    new_df = pd.DataFrame()
    word_map = torch.load(map_path)
    
    label_maps = []
    label_lengths = []
   
    df = pd.read_csv(csv_path)
    
    if TASK_NAME == 'Drug_like':
        df['file_path'] = df['file_path'].apply(lambda x: os.path.join('/home', 'rlawlsgurjh', 'work', 'MolScribe', 'data', x))
     
    # 컬럼 변환
    new_df['image_path'] = df['file_path']  # 또는 df['file_path'], 상황에 맞게 수정
    new_df['label'] = df['SMILES']

    for smiles in tqdm(df['SMILES'], desc="Creating label maps"):
        label_map, length = create_label_map(smiles, word_map, max_length)
        label_maps.append(label_map)
        label_lengths.append(length)
    
    new_df['label_map'] = label_maps
    new_df['label_length'] = label_lengths
    
    new_df.to_pickle(save_path)
    
    return new_df

In [48]:
pkl_df = convert_to_pickle(TASK_NAME, csv_path, map_path, save_path)
print(pkl_df.info())
print(pkl_df.head())

Creating label maps: 100%|██████████| 13020/13020 [00:00<00:00, 455032.69it/s]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13020 entries, 0 to 13019
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   image_path    13020 non-null  object
 1   label         13020 non-null  object
 2   label_map     13020 non-null  object
 3   label_length  13020 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 407.0+ KB
None
                                          image_path  \
0  /home/rlawlsgurjh/work/MolScribe/data/DECIMER/...   
1  /home/rlawlsgurjh/work/MolScribe/data/DECIMER/...   
2  /home/rlawlsgurjh/work/MolScribe/data/DECIMER/...   
3  /home/rlawlsgurjh/work/MolScribe/data/DECIMER/...   
4  /home/rlawlsgurjh/work/MolScribe/data/DECIMER/...   

                             label  \
0               CCC1=C=CC2(C)PC12C   
1                C1=NC=C2N=CPC2=N1   
2                     C=CCN=C(N)SN   
3   O=C1NC(=O)C(=COCC2=CC=CC=C2)N1   
4  CCC1=NC(C(=N)N2CCNC(C)C2=N)=CS1   

                 